# `expansion_hunter_02_run_batch` — native Google Batch

Run **in a VWB JupyterLab app** after
[`expansion_hunter_00_prep_reference.ipynb`](expansion_hunter_00_prep_reference.ipynb).

This submits **minicram** with `gcloud batch` (one print-reads container).
No dsub sidecars. Proven in this workspace:

- host `gcloud storage cp` (`eh-gcs-smoke`)
- in-container `google.cloud.storage` (`eh-ctr-smoke-260929-034427`)

The WDL notebook ([`expansion_hunter_01_run.ipynb`](expansion_hunter_01_run.ipynb))
is unchanged. Docs: [`expansion_hunter/batch/README.md`](../../expansion_hunter/batch/README.md).

## What this notebook does

1. Check `PET_SA_EMAIL`, `GOOGLE_CLOUD_PROJECT`, `gcloud`
2. Write a per-sample CSV (same columns as `configs/batch.header.csv`)
3. Build the Batch job JSON (dry-run; always)
4. Optionally submit minicram (`SUBMIT_MINICRAM`)
5. Describe the job and list GCS outputs

`SUBMIT_MINICRAM` stays **off** until you turn it on. Genotype is not wired
yet — run it after the minicram objects exist.

The WGS CRAM stays a `gs://` env var so Batch never localizes the Nearline
object. The worker downloads catalog, hg38, and the CRAI with the GCS
client, then range-fetches CRAM containers. Do not call `gcloud` inside
print-reads `0.1.2` (`gcloud`'s shebang is bash; slim has no bash → exit 127).


In [ ]:
from __future__ import annotations

import csv
import json
import os
import shutil
import subprocess
import sys
from datetime import datetime, timezone
from pathlib import Path


def find_repo_root() -> Path:
    here = Path.cwd().resolve()
    for p in [here, *here.parents]:
        script = p / "expansion_hunter" / "batch" / "submit_batch.py"
        if script.is_file():
            return p
        nested = p / "aou-lr-phase-2" / "expansion_hunter" / "batch" / "submit_batch.py"
        if nested.is_file():
            return nested.parents[2]
    raise FileNotFoundError(
        "Cannot find expansion_hunter/batch/submit_batch.py. Clone "
        "kvg/aou-lr-phase-2 into this app (or cd into the clone) and re-run."
    )


def sh(cmd: list[str], *, check: bool = True) -> subprocess.CompletedProcess[str]:
    print("$", " ".join(cmd), flush=True)
    return subprocess.run(cmd, check=check, text=True)


REPO_ROOT = find_repo_root()
BATCH_DIR = REPO_ROOT / "expansion_hunter" / "batch"
WORKER_PY = BATCH_DIR / "make_minicram.py"
SUBMIT_PY = BATCH_DIR / "submit_batch.py"
HEADER_CSV = REPO_ROOT / "expansion_hunter" / "configs" / "batch.header.csv"
SCRATCH = Path.cwd() / "expansion_hunter_rw_scratch"
SCRATCH.mkdir(parents=True, exist_ok=True)

sys.path.insert(0, str(BATCH_DIR))
from submit_batch import build_minicram_job  # noqa: E402

OUTPUT_BUCKET_GS = os.environ.get("EH_OUTPUT_BUCKET_GS", "gs://aou-lr-phase2-resources/")
OUT_PREFIX = os.environ.get(
    "EH_BATCH_OUT_PREFIX",
    f"{OUTPUT_BUCKET_GS.rstrip('/')}/batchRuns/expansion_hunter",
)
REGION = os.environ.get("DSUB_REGION", "us-central1")

SAMPLE_ID = os.environ.get("EH_SAMPLE_ID", "1000000")
CRAM = os.environ.get(
    "EH_CRAM",
    "gs://vwb-aou-datasets-controlled/pooled/wgs/cram/v8_base/wgs_1000000.cram",
)
CRAI = os.environ.get(
    "EH_CRAI",
    "gs://vwb-aou-datasets-controlled/pooled/wgs/cram/v8_base/wgs_1000000.cram.crai",
)
REF_FA = os.environ.get(
    "EH_REF_FA",
    "gs://aou-lr-phase2-resources/locityper/refs/Homo_sapiens_assembly38.fasta",
)
REF_FAI = os.environ.get(
    "EH_REF_FAI",
    "gs://aou-lr-phase2-resources/locityper/refs/Homo_sapiens_assembly38.fasta.fai",
)
CATALOG = os.environ.get(
    "EH_CATALOG",
    "gs://aou-lr-phase2-resources/expansion_hunter/smoke.catalog.json",
)
SEX = os.environ.get("EH_SEX", "female")
GCLOUD_PROJECT = os.environ.get("EH_GCLOUD_PROJECT", os.environ.get("GOOGLE_CLOUD_PROJECT", ""))

PRINT_READS_DOCKER = os.environ.get(
    "EH_PRINT_READS_DOCKER",
    "us-central1-docker.pkg.dev/broad-dsp-lrma/aou-lr/aou-locityper-print-reads:0.1.2",
)

# Optional path to a local batch CSV (same columns as batch.header.csv).
# If empty, the next cell writes a one-row CSV from SAMPLE_ID / CRAM / … .
BATCH_CSV_LOCAL = os.environ.get("EH_BATCH_CSV", "")

SUBMIT_MINICRAM = False
WATCH_JOB_ID = ""  # after submit, or leave empty to use last_minicram.job_id
JOB_ID_PATH = SCRATCH / "last_minicram.job_id"

print("REPO_ROOT:", REPO_ROOT)
print("WORKER_PY:", WORKER_PY, "exists=" + str(WORKER_PY.is_file()))
print("SUBMIT_PY:", SUBMIT_PY, "exists=" + str(SUBMIT_PY.is_file()))
print("OUTPUT_BUCKET_GS:", OUTPUT_BUCKET_GS)
print("OUT_PREFIX:", OUT_PREFIX)
print("SAMPLE_ID:", SAMPLE_ID or "(unset)")
print("CRAM:", CRAM)
print("CATALOG:", CATALOG or "(unset)")
print("PRINT_READS_DOCKER:", PRINT_READS_DOCKER)
print("GCLOUD_PROJECT:", GCLOUD_PROJECT or "(unset)")
print("PET_SA_EMAIL:", os.environ.get("PET_SA_EMAIL", "(unset)"))
print("SUBMIT_MINICRAM:", SUBMIT_MINICRAM)
print("gcloud:", shutil.which("gcloud"))


## PET SA / VPC

Jobs run as `PET_SA_EMAIL` on the workspace private VPC (`network` /
`subnetwork` in `us-central1`), same allocation as the successful container
smoke. No `dsub`. PET cannot read Cloud Logging; after SUCCESS read
`{sample}.minicram.worker.log` in GCS.


In [ ]:
if shutil.which("gcloud") is None:
    raise SystemExit("gcloud is not on PATH. Open this notebook in a VWB Jupyter app.")
if not os.environ.get("GOOGLE_CLOUD_PROJECT"):
    raise SystemExit("GOOGLE_CLOUD_PROJECT is empty. Open this notebook in a VWB Jupyter app.")
if not os.environ.get("PET_SA_EMAIL"):
    raise SystemExit("PET_SA_EMAIL is empty. VWB sets this in cloud apps.")
if not WORKER_PY.is_file():
    raise SystemExit(f"missing worker: {WORKER_PY}")

sh(["gcloud", "config", "get-value", "project"], check=False)
print("project:", os.environ["GOOGLE_CLOUD_PROJECT"])
print("PET_SA_EMAIL:", os.environ["PET_SA_EMAIL"])
print("network: projects/{}/global/networks/network".format(os.environ["GOOGLE_CLOUD_PROJECT"]))
print("subnetwork: projects/{}/regions/{}/subnetworks/subnetwork".format(
    os.environ["GOOGLE_CLOUD_PROJECT"], REGION
))


## Batch CSV

Same columns as [`expansion_hunter/configs/batch.header.csv`](../../expansion_hunter/configs/batch.header.csv).
The WGS CRAM **and** CRAI stay `gs://` (never localized). This cell always
runs; it does not submit.


In [ ]:
csv_path = Path(BATCH_CSV_LOCAL) if BATCH_CSV_LOCAL else SCRATCH / "expansion_hunter.batch.csv"
if BATCH_CSV_LOCAL:
    if not csv_path.is_file():
        raise SystemExit(f"BATCH_CSV_LOCAL does not exist: {csv_path}")
    print("using existing CSV:", csv_path)
else:
    required = {
        "SAMPLE_ID": SAMPLE_ID,
        "CRAM": CRAM,
        "CRAI": CRAI,
        "REF_FA": REF_FA,
        "REF_FAI": REF_FAI,
        "CATALOG": CATALOG,
    }
    missing = [k for k, v in required.items() if not v]
    if missing:
        raise SystemExit(f"fill {missing} in the config cell (or set BATCH_CSV_LOCAL)")
    fieldnames = ["sample_id", "cram", "crai", "ref_fa", "ref_fai", "catalog", "sex", "gcloud_project"]
    csv_path.parent.mkdir(parents=True, exist_ok=True)
    with csv_path.open("w", newline="") as fh:
        w = csv.DictWriter(fh, fieldnames=fieldnames)
        w.writeheader()
        w.writerow(
            {
                "sample_id": SAMPLE_ID,
                "cram": CRAM,
                "crai": CRAI,
                "ref_fa": REF_FA,
                "ref_fai": REF_FAI,
                "catalog": CATALOG,
                "sex": SEX,
                "gcloud_project": GCLOUD_PROJECT,
            }
        )
    print("wrote one-row CSV:", csv_path)

print(csv_path.read_text())


## Build Batch JSON (dry-run)

Always runs. Does not submit. The worker script is embedded as
`python3 -c …` so the published `0.1.2` image does not need a rebuild.


In [ ]:
project = os.environ["GOOGLE_CLOUD_PROJECT"]
sa = os.environ["PET_SA_EMAIL"]
job = build_minicram_job(
    csv_path=csv_path,
    worker_path=WORKER_PY,
    image=PRINT_READS_DOCKER,
    out_prefix=OUT_PREFIX,
    project=project,
    region=REGION,
    sa=sa,
)
cfg_path = SCRATCH / "minicram.batch.json"
cfg_path.write_text(json.dumps(job, indent=2) + "\n", encoding="utf-8")

group = job["taskGroups"][0]
spec = group["taskSpec"]
env = dict(spec["environment"]["variables"])
worker_len = len(spec["runnables"][0]["container"]["commands"][1])
print("config:", cfg_path)
print("tasks:", group["taskCount"])
print("image:", spec["runnables"][0]["container"]["imageUri"])
print("entrypoint:", spec["runnables"][0]["container"]["entrypoint"])
print("worker bytes:", worker_len)
print("cpuMilli:", spec["computeResource"]["cpuMilli"], "memoryMib:", spec["computeResource"]["memoryMib"])
print("bootDiskMib:", spec["computeResource"]["bootDiskMib"])
print("maxRunDuration:", spec["maxRunDuration"])
print("noExternalIp:", job["allocationPolicy"]["network"]["networkInterfaces"][0]["noExternalIpAddress"])
print("serviceAccount:", job["allocationPolicy"]["serviceAccount"]["email"])
print("SAMPLE_ID:", env.get("SAMPLE_ID"))
print("CRAM:", env.get("CRAM"))
print("CRAI:", env.get("CRAI"))
print("MINICRAM:", env.get("MINICRAM"))
print("WORKER_LOG_GCS:", env.get("WORKER_LOG_GCS"))
if "taskEnvironments" in group:
    print("per-task samples:", [e["variables"]["SAMPLE_ID"] for e in group["taskEnvironments"]])


## Submit minicram

Set `SUBMIT_MINICRAM = True` in the config cell and re-run from there.
First sample is slower than the 2s GCS smoke: new VM, image pull, ~3 GB
fasta, CRAI, then catalog-window CRAM ranges (minutes, not seconds).


In [ ]:
if not SUBMIT_MINICRAM:
    print("SUBMIT_MINICRAM is off; not submitting. Flip it in the config cell and re-run.")
else:
    job_id = "eh-mini-" + datetime.now(timezone.utc).strftime("%y%m%d-%H%M%S")
    sh(
        [
            "gcloud",
            "batch",
            "jobs",
            "submit",
            job_id,
            "--location",
            REGION,
            "--project",
            os.environ["GOOGLE_CLOUD_PROJECT"],
            "--config",
            str(cfg_path),
        ]
    )
    JOB_ID_PATH.write_text(job_id + "\n", encoding="utf-8")
    print("JOB_ID=" + job_id)
    print("wrote", JOB_ID_PATH)
    print("watch:")
    print(
        f"  gcloud batch jobs describe {job_id} --location={REGION} "
        "--format='yaml(status.state,status.statusEvents)'"
    )


## Status / outputs

PET cannot read Cloud Logging. On SUCCESS, list the sample prefix and cat
the worker log. Set `WATCH_JOB_ID` in the config cell, or this cell uses
`last_minicram.job_id` from submit.


In [ ]:
job_id = (WATCH_JOB_ID or "").strip()
if not job_id and JOB_ID_PATH.is_file():
    job_id = JOB_ID_PATH.read_text(encoding="utf-8").strip()

print("OUT_PREFIX:", OUT_PREFIX)
print("expected outputs:")
prefix = f"{OUT_PREFIX.rstrip('/')}/{SAMPLE_ID}/{SAMPLE_ID}"
for suffix in (
    ".minicram.cram",
    ".minicram.cram.crai",
    ".data_transfer_stats.tsv",
    ".minicram.worker.log",
):
    print(" ", prefix + suffix)

if not job_id:
    print("no JOB_ID yet (submit first, or set WATCH_JOB_ID)")
else:
    print("JOB_ID:", job_id)
    sh(
        [
            "gcloud",
            "batch",
            "jobs",
            "describe",
            job_id,
            "--location",
            REGION,
            "--project",
            os.environ["GOOGLE_CLOUD_PROJECT"],
            "--format=yaml(status.state,status.statusEvents)",
        ],
        check=False,
    )
    sample_dir = f"{OUT_PREFIX.rstrip('/')}/{SAMPLE_ID}/"
    print("listing", sample_dir)
    sh(["gcloud", "storage", "ls", sample_dir], check=False)
